<a href="https://colab.research.google.com/github/SravaniPurra/Poject-of-AI-ML/blob/main/1_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import re
import os

In [2]:
BASE_URL = "https://books.toscrape.com/"

def get_soup(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

In [3]:
soup = get_soup(BASE_URL)
category_links = []
for link in soup.select(".side_categories ul li ul li a"):
    category_name = link.text.strip()
    category_url = BASE_URL + link["href"]
    category_links.append({
        "category": category_name,
        "url": category_url
    })
categories_df = pd.DataFrame(category_links)
print(categories_df.head())

             category                                                url
0              Travel  https://books.toscrape.com/catalogue/category/...
1             Mystery  https://books.toscrape.com/catalogue/category/...
2  Historical Fiction  https://books.toscrape.com/catalogue/category/...
3      Sequential Art  https://books.toscrape.com/catalogue/category/...
4            Classics  https://books.toscrape.com/catalogue/category/...


In [4]:
selected_categories = categories_df.head(3)
print(selected_categories)

             category                                                url
0              Travel  https://books.toscrape.com/catalogue/category/...
1             Mystery  https://books.toscrape.com/catalogue/category/...
2  Historical Fiction  https://books.toscrape.com/catalogue/category/...


In [5]:
books = []

for _, category in selected_categories.iterrows():

    category_name = category["category"]
    url = category["url"]

    while url:

        soup = get_soup(url)

        for book in soup.select("article.product_pod"):

            title = book.h3.a["title"]

            price = book.select_one(".price_color").text.strip()

            rating = book.select_one(".star-rating")["class"][1]

            availability = book.select_one(".availability").get_text(" ", strip=True)

            books.append({
                "title": title,
                "price": price,
                "star_rating": rating,
                "availability": availability,
                "category": category_name
            })

        # Find next page
        next_button = soup.select_one("li.next a")

        if next_button:
            url = url.rsplit("/", 1)[0] + "/" + next_button["href"]
        else:
            url = None

print("Total books scraped:", len(books))

Total books scraped: 69


In [6]:
df = pd.DataFrame(books)
print(df.head())
print("\nShape:", df.shape)

                                               title    price star_rating  \
0                            It's Only the Himalayas  Â£45.17         Two   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...  Â£49.43        Four   
2  See America: A Celebration of Our National Par...  Â£48.87       Three   
3  Vagabonding: An Uncommon Guide to the Art of L...  Â£36.94         Two   
4                               Under the Tuscan Sun  Â£37.33       Three   

  availability category  
0     In stock   Travel  
1     In stock   Travel  
2     In stock   Travel  
3     In stock   Travel  
4     In stock   Travel  

Shape: (69, 5)


In [8]:
def clean_price(value):
    try:
        value = re.sub(r"[^0-9.]", "", value)
        return float(value)
    except:
        return None

df["price_gbp"] = df["price"].apply(clean_price)

In [9]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

In [11]:
def clean_stock(value):
    value = value.lower()
    if "in stock" in value:
        return True
    elif "out of stock" in value:
        return False
    else:
        return None
df["in_stock"] = df["availability"].apply(clean_stock)

In [12]:
df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())
df["rating"] = df["rating"].fillna(df["rating"].median()).astype(int)

In [13]:
df = df.dropna(subset=["title", "category", "in_stock"])
print("Cleaned rows:", len(df))

Cleaned rows: 69


In [15]:
GBP_TO_INR = 105.50
df["price_inr"] = df["price_gbp"] * GBP_TO_INR
print(df[["title", "price_gbp", "price_inr"]].head())

                                               title  price_gbp  price_inr
0                            It's Only the Himalayas      45.17   4765.435
1  Full Moon over Noahâs Ark: An Odyssey to Mou...      49.43   5214.865
2  See America: A Celebration of Our National Par...      48.87   5155.785
3  Vagabonding: An Uncommon Guide to the Art of L...      36.94   3897.170
4                               Under the Tuscan Sun      37.33   3938.315


In [17]:
final_df = df[
    ["title",
     "price_gbp",
      "price_inr",
      "rating",
      "in_stock",
      "category" ]].copy()
print(final_df.head())
print("\nShape:", final_df.shape)
print("\nData types:")
print(final_df.dtypes)

                                               title  price_gbp  price_inr  \
0                            It's Only the Himalayas      45.17   4765.435   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...      49.43   5214.865   
2  See America: A Celebration of Our National Par...      48.87   5155.785   
3  Vagabonding: An Uncommon Guide to the Art of L...      36.94   3897.170   
4                               Under the Tuscan Sun      37.33   3938.315   

   rating  in_stock category  
0       2      True   Travel  
1       4      True   Travel  
2       3      True   Travel  
3       2      True   Travel  
4       3      True   Travel  

Shape: (69, 6)

Data types:
title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object


In [18]:
conn = sqlite3.connect("zepto_books.db")
cursor = conn.cursor()

In [19]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

conn.commit()

print("Tables created successfully")

Tables created successfully


In [21]:
for category in final_df["category"].unique():
  cursor.execute(
        "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
        (category,))
conn.commit()
print("Categories inserted")

Categories inserted


In [22]:
for _, row in final_df.iterrows():

    cursor.execute("""
        SELECT category_id
        FROM categories
        WHERE category_name = ?
    """, (row["category"],))

    category_id = cursor.fetchone()[0]

    cursor.execute("""
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

print("Books inserted:", len(final_df))

Books inserted: 69


In [23]:
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4;
"""
result1 = pd.read_sql(query1, conn)
print(result1)

                                                title  price_gbp  rating
0   Full Moon over Noahâs Ark: An Odyssey to Mou...      49.43       4
1                    A Year in Provence (Provence #1)      56.88       4
2                  1,000 Places to See Before You Die      26.08       5
3                                       Sharp Objects      47.82       4
4                                 The Past Never Ends      56.50       4
5     The Murder of Roger Ackroyd (Hercule Poirot #4)      44.10       4
6              A Time of Torment (Charlie Parker #14)      48.35       5
7   Murder at the 42nd Street Library (Raymond Amb...      54.36       4
8   What Happened on Beale Street (Secrets of the ...      25.37       5
9   The Bachelor Girl's Guide to Murder (Herringfo...      52.30       5
10   Delivering the Truth (Quaker Midwife Mystery #1)      20.89       4
11  The Mysterious Affair at Styles (Hercule Poiro...      24.80       4
12                  The Silkworm (Cormoran Strike #

In [24]:
query2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""
result2 = pd.read_sql(query2, conn)
print(result2)

                                               title  price_gbp  rating
0                      Boar Island (Anna Pigeon #19)      59.48       3
1  The No. 1 Ladies' Detective Agency (No. 1 Ladi...      57.70       4
2                   A Year in Provence (Provence #1)      56.88       4
3                                The Past Never Ends      56.50       4
4                   The Last Painting of Sara de Vos      55.55       2
5            A Flight of Arrows (The Pathfinders #2)      55.53       5
6  Murder at the 42nd Street Library (Raymond Amb...      54.36       4
7                     The Last Mile (Amos Decker #2)      54.21       2
8                1st to Die (Women's Murder Club #1)      53.98       1
9                                 Tipping the Velvet      53.74       1


In [26]:
query3 = """
SELECT DISTINCT category_name
FROM categories;
"""
result3 = pd.read_sql(query3, conn)
print(result3)

        category_name
0  Historical Fiction
1             Mystery
2              Travel


In [28]:
query4 = """
SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp BETWEEN 20 AND 40;
"""
result4 = pd.read_sql(query4, conn)
print(result4)

                                                title  price_gbp  price_inr
0   Vagabonding: An Uncommon Guide to the Art of L...      36.94   3897.170
1                                Under the Tuscan Sun      37.33   3938.315
2                            The Great Railway Bazaar      30.54   3221.970
3   The Road to Little Dribbling: Adventures of an...      23.21   2448.655
4           Neither Here nor There: Travels in Europe      38.95   4109.225
5                  1,000 Places to See Before You Die      26.08   2751.440
6                    Poisonous (Max Revere Novels #3)      26.80   2827.400
7                                         Most Wanted      35.28   3722.040
8                                           The Widow      27.26   2875.930
9   What Happened on Beale Street (Secrets of the ...      25.37   2676.535
10   Delivering the Truth (Quaker Midwife Mystery #1)      20.89   2203.895
11  The Mysterious Affair at Styles (Hercule Poiro...      24.80   2616.400
12          

In [30]:
query5 = """
SELECT title, rating, category_id
FROM books
WHERE rating IN (4, 5);
"""
result5 = pd.read_sql(query5, conn)
print(result5)

                                                title  rating  category_id
0   Full Moon over Noahâs Ark: An Odyssey to Mou...       4            1
1                    A Year in Provence (Provence #1)       4            1
2                  1,000 Places to See Before You Die       5            1
3                                       Sharp Objects       4            2
4                                 The Past Never Ends       4            2
5     The Murder of Roger Ackroyd (Hercule Poirot #4)       4            2
6              A Time of Torment (Charlie Parker #14)       5            2
7   Murder at the 42nd Street Library (Raymond Amb...       4            2
8   What Happened on Beale Street (Secrets of the ...       5            2
9   The Bachelor Girl's Guide to Murder (Herringfo...       5            2
10   Delivering the Truth (Quaker Midwife Mystery #1)       4            2
11  The Mysterious Affair at Styles (Hercule Poiro...       4            2
12                  The S

In [31]:
query6 = """
SELECT
    b.title,
    c.category_name,
    b.rating,
    b.price_gbp,
    b.price_inr,
    b.in_stock
FROM books b
JOIN categories c
ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;
"""
result6 = pd.read_sql(query6, conn)
print(result6)

                                               title       category_name  \
0            A Flight of Arrows (The Pathfinders #2)  Historical Fiction   
1  The Bachelor Girl's Guide to Murder (Herringfo...             Mystery   
2             A Time of Torment (Charlie Parker #14)             Mystery   
3                                While You Were Mine  Historical Fiction   
4                                       The Red Tent  Historical Fiction   
5                                       Mrs. Houdini  Historical Fiction   
6                              The Passion of Dolssa  Historical Fiction   
7                 1,000 Places to See Before You Die              Travel   
8  What Happened on Beale Street (Secrets of the ...             Mystery   
9                  The Silkworm (Cormoran Strike #2)             Mystery   

   rating  price_gbp  price_inr  in_stock  
0       5      55.53   5858.415         1  
1       5      52.30   5517.650         1  
2       5      48.35   5100.925

In [33]:
queries = {
    "query1_where": (query1, result1),
    "query2_order_limit": (query2, result2),
    "query3_distinct": (query3, result3),
    "query4_between": (query4, result4),
    "query5_in": (query5, result5),
    "query6_join": (query6, result6)
}
for name, (query, result) in queries.items():

    print("\n" + "*" * 50)
    print(name)
    print("*" * 50)

    print("SQL:")
    print(query)

    print("OUTPUT:")
    print(result)


query1_where
SQL:

SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4;

OUTPUT:
                                                title  price_gbp  rating
0   Full Moon over Noahâs Ark: An Odyssey to Mou...      49.43       4
1                    A Year in Provence (Provence #1)      56.88       4
2                  1,000 Places to See Before You Die      26.08       5
3                                       Sharp Objects      47.82       4
4                                 The Past Never Ends      56.50       4
5     The Murder of Roger Ackroyd (Hercule Poirot #4)      44.10       4
6              A Time of Torment (Charlie Parker #14)      48.35       5
7   Murder at the 42nd Street Library (Raymond Amb...      54.36       4
8   What Happened on Beale Street (Secrets of the ...      25.37       5
9   The Bachelor Girl's Guide to Murder (Herringfo...      52.30       5
10   Delivering the Truth (Quaker Midwife Mystery #1)      20.89       4
11  The Mysterious Affair at Styl

In [34]:
for name, (query, result) in queries.items():
    result.to_csv(name + ".csv", index=False)
print("All query outputs saved.")

All query outputs saved.


In [36]:
df_sql1 = pd.read_sql(query1, conn)
df_sql2 = pd.read_sql(query2, conn)
print("First SQL result:")
print(df_sql1.head())
print("\nSecond SQL result:")
print(df_sql2.head())

First SQL result:
                                               title  price_gbp  rating
0  Full Moon over Noahâs Ark: An Odyssey to Mou...      49.43       4
1                   A Year in Provence (Provence #1)      56.88       4
2                 1,000 Places to See Before You Die      26.08       5
3                                      Sharp Objects      47.82       4
4                                The Past Never Ends      56.50       4

Second SQL result:
                                               title  price_gbp  rating
0                      Boar Island (Anna Pigeon #19)      59.48       3
1  The No. 1 Ladies' Detective Agency (No. 1 Ladi...      57.70       4
2                   A Year in Provence (Provence #1)      56.88       4
3                                The Past Never Ends      56.50       4
4                   The Last Painting of Sara de Vos      55.55       2


In [37]:
books_df = pd.read_sql("SELECT * FROM books", conn)
categories_df = pd.read_sql("SELECT * FROM categories", conn)
print("Books:")
print(books_df.head())
print("\nCategories:")
print(categories_df.head())

Books:
   book_id                                              title  price_gbp  \
0        1                            It's Only the Himalayas      45.17   
1        2  Full Moon over Noahâs Ark: An Odyssey to Mou...      49.43   
2        3  See America: A Celebration of Our National Par...      48.87   
3        4  Vagabonding: An Uncommon Guide to the Art of L...      36.94   
4        5                               Under the Tuscan Sun      37.33   

   price_inr  rating  in_stock  category_id  
0   4765.435       2         1            1  
1   5214.865       4         1            1  
2   5155.785       3         1            1  
3   3897.170       2         1            1  
4   3938.315       3         1            1  

Categories:
   category_id       category_name
0            1              Travel
1            2             Mystery
2            3  Historical Fiction


In [40]:
merged_df = pd.merge(books_df,categories_df,
    on="category_id",
    how="inner")
merged_df = merged_df[
    ["title",
     "category_name",
     "rating",
     "price_gbp",
     "price_inr",
     "in_stock"]]
merged_df = merged_df.sort_values(by=["rating", "price_gbp"],
    ascending=[False, False]).head(10)
print(merged_df)

                                                title       category_name  \
45            A Flight of Arrows (The Pathfinders #2)  Historical Fiction   
29  The Bachelor Girl's Guide to Murder (Herringfo...             Mystery   
19             A Time of Torment (Charlie Parker #14)             Mystery   
64                                While You Were Mine  Historical Fiction   
59                                       The Red Tent  Historical Fiction   
47                                       Mrs. Houdini  Historical Fiction   
56                              The Passion of Dolssa  Historical Fiction   
10                 1,000 Places to See Before You Die              Travel   
28  What Happened on Beale Street (Secrets of the ...             Mystery   
33                  The Silkworm (Cormoran Strike #2)             Mystery   

    rating  price_gbp  price_inr  in_stock  
45       5      55.53   5858.415         1  
29       5      52.30   5517.650         1  
19       5      4

In [42]:
sql_join = result6.reset_index(drop=True)
pandas_join = merged_df.reset_index(drop=True)
print("SQL JOIN result:")
print(sql_join)
print("\nPandas merge result:")
print(pandas_join)

SQL JOIN result:
                                               title       category_name  \
0            A Flight of Arrows (The Pathfinders #2)  Historical Fiction   
1  The Bachelor Girl's Guide to Murder (Herringfo...             Mystery   
2             A Time of Torment (Charlie Parker #14)             Mystery   
3                                While You Were Mine  Historical Fiction   
4                                       The Red Tent  Historical Fiction   
5                                       Mrs. Houdini  Historical Fiction   
6                              The Passion of Dolssa  Historical Fiction   
7                 1,000 Places to See Before You Die              Travel   
8  What Happened on Beale Street (Secrets of the ...             Mystery   
9                  The Silkworm (Cormoran Strike #2)             Mystery   

   rating  price_gbp  price_inr  in_stock  
0       5      55.53   5858.415         1  
1       5      52.30   5517.650         1  
2       5     

In [43]:
same_result = sql_join.equals(pandas_join)
print("Are SQL JOIN and pandas merge equivalent?", same_result)

Are SQL JOIN and pandas merge equivalent? True


In [44]:
final_df.to_csv("cleaned_books.csv", index=False)
print("cleaned_books.csv saved successfully")

cleaned_books.csv saved successfully


In [45]:
conn.close()
print("Database connection closed")

Database connection closed
